# Risk Score Pipeline — `amygda_ops_risk_score`

## What this notebook does

Continues from `01_labelling.ipynb`.  Takes the **trained labelling model zip**
produced by `run_classification()` and trains a risk score model, then generates
a `risk_scores.parquet` file with per-asset, per-period operational risk scores
across every system and subsystem in the hierarchy.

This notebook covers **both training modes** — choose one by setting `SUPERVISED`:

| Mode | `SUPERVISED` | Calibration method | Weights source | Requires |
|------|--------------|--------------------|---------------|----------|
| **Unsupervised** | `False` | Quantile of training distribution | Expert weights from labelling | Training log file only |
| **Supervised** | `True` | AUC-ROC grid search over threshold candidates | Logistic regression fitted on failure labels | Training log file + failures CSV |

### How unsupervised calibration works

For each subsystem the server computes the distribution of `_risk_feature` values across
all assets and dates in the training set.  The calibration threshold is set at the
`quantile_for_thresholds` percentile (default 0.99 — top 1 %).
Any day where a subsystem's activity exceeds the threshold is considered elevated risk.
Weights (how much each system contributes to the overall operational risk score) come
directly from the expert weights defined during the labelling step.

### How supervised calibration differs

1. **Threshold learning** — instead of a fixed percentile, the API searches for the
   threshold that maximises AUC-ROC against your labelled events.  Subsystems with
   too few positive examples fall back to the quantile approach automatically.
2. **Weight learning** — a logistic regression is fitted on the calibrated features
   to learn which systems and subsystems are most predictive of failures.  Expert
   weights are used as a fallback for any system that receives a zero coefficient.

## What you need before starting

- Your permanent `api_key` (same key used in `01_labelling.ipynb`)
- The `trained_labelling_model_ses-xxx.zip` downloaded by `run_classification()`
- *(Supervised only)* A failures CSV with at least two columns:
  - an asset identifier column (matching `asset_id_column` in your log file)
  - a failure date column

The labelling artifacts are loaded automatically from `LABELLING_ARTIFACT_DIR` —
no copy-pasting of keys or paths needed.

## Step overview

| Step | Method | Re-runnable? | Locked after |
|------|--------|--------------|-------------|
| 9  | `configure_training` | Yes | `train_risk_model` starts |
| 10 | `train_risk_model` | Yes | `configure_generation` starts |
| 11 | `configure_generation` | Yes | `generate_risk_scores` starts |
| 12 | `generate_risk_scores(dest_dir)` | **ONE-WAY DOOR** | — |

## Which zip to pass to `import_model()`

| Zip type | Unlocks |
|----------|---------|
| `trained_labelling_model_ses-xxx.zip` | Steps 9–12 (full training required) |
| `complete_model_ses-xxx.zip` | Step 11 only (thresholds already trained — skip to generation) |

## Data modes — free-text vs fixed-log

The SDK reads `model_config.json` to determine which mode your session uses.
You do not need to specify it manually.

| Mode | How logs are classified | `rolling_feature_type` support |
|------|------------------------|--------------------------------|
| **Free-text** | Raw log messages classified semantically via Qdrant vector search | `'sum'` and `'flag'` only |
| **Fixed-log** | Log codes matched exactly against a pre-defined dictionary | `'sum'`, `'flag'`, `'ewm'`, `'all'` |

Supervised calibration works identically for both modes — it operates on the
`_risk_feature` columns produced by feature engineering, which have the same
structure regardless of the upstream classification method.

## Artifacts written to `ARTIFACT_DIR` during this notebook

```
artifacts/risk_score/
  configure_training.json           ← saved by configure_training()
  train_risk_model.json             ← saved by train_risk_model()
  calibration_thresholds.json       ← quantile-based (unsupervised) or AUC-optimised (supervised)
  training_fe.parquet               ← feature-engineered training dataset
  training_scores.parquet           ← training data scored with trained model
  trained_weights.json              ← trained weights (supervised only)
  supervised_training_report.json   ← AUC-ROC / avg-precision baseline vs supervised (supervised only)
  logs_by_system_subsystem.json     ← log-to-subsystem mapping (fixed-log only)
  configure_generation.json         ← saved by configure_generation()
  generate_risk_scores.json         ← saved by generate_risk_scores()
  risk_scores.parquet               ← final risk scores — auto-extracted from zip
  classified_logs.parquet           ← classified log entries (free-text only)
  accepted_hierarchy.json           ← hierarchy used for scoring
  model_config.json                 ← labelling expert weights and session config
```

## Session lifecycle guide

Read this once — the rules are the same for every risk score session.

---

### Every time you open this notebook — run the imports cell first

---

### API readiness — always call `client.wait_until_ready()` before anything else

The API uses scale-to-zero hosting.  On a cold start the ML models take 60–120 s to load.
`wait_until_ready()` polls silently until the API responds and returns immediately if it is
already warm.  Never skip this call.

---

### Case 1 — Starting a new session

Run the setup cells in order: *Imports → Connect → Load credentials → Open session*.
The `api_key` is the same permanent key from `01_labelling.ipynb` — no re-registration needed.

---

### Case 2 — Kernel restarted mid-way through

Run the *Reconnect* cell further below.  Paste your `AUTH_ID` and `API_KEY` then call
`client.open_session()` — it auto-resumes the existing session if it is still alive.
If it raises `SessionNotFoundError` the session TTL has expired: re-run the full setup
and call `session.import_model(ZIP_PATH)` again.

---

### Case 3 — Network dropped and came back

Your in-memory `session` object is still intact — just continue from where you stopped.
If `session.status()` raises `APIError (404)` call `client.open_session()` to get a
fresh handle; it resumes automatically if the session is still alive.  **No re-upload needed.**

---

### Case 4 — A step is locked and you need to go back

Steps are locked once their successor starts (see the table in the intro).  To unlock:

```python
session = client.restart_session(session, api_key=API_KEY, config=config)
# All step locks are cleared.  Re-run from import_model() onwards.
```

⚠ This permanently deletes server-side session data.  Your `api_key` is never affected.

---

### The `next_step` key

Every step method returns a dict that includes `next_step` — the exact method name to
call next.  Use it to confirm you are on the right track:

```python
result = session.configure_training(...)
print(result['next_step'])  # → 'train_risk_model'
```

| Step completed | `next_step` value |
|---|---|
| `import_model` | `configure_training` |
| `configure_training` | `train_risk_model` |
| `train_risk_model` | `configure_generation` |
| `configure_generation` | `generate_risk_scores` |
| `generate_risk_scores` | `done` |

In [ ]:

import json, sys, os

from amygda_ops_risk_score import OpsRiskClient, SessionConfig
from amygda_ops_risk_score import helpers

print("SDK imported successfully.")

In [ ]:
# ── Connect to API ────────────────────────────────────────────────────────────
# For Cloud Run deployments: OpsRiskClient(base_url="https://your-service-url")
client = OpsRiskClient()
client.wait_until_ready()
print("API is ready.")

In [ ]:
# ── Paths — edit once ─────────────────────────────────────────────────────────
LABELLING_ARTIFACT_DIR = "artifacts/labelling/"    # output folder from 01_labelling.ipynb
ARTIFACT_DIR           = "artifacts/risk_score/"   # risk score artifacts land here

with open(f"{LABELLING_ARTIFACT_DIR}api_key.txt") as f:
    API_KEY = f.read().strip()

with open(f"{LABELLING_ARTIFACT_DIR}run_classification.json") as f:
    ZIP_PATH = json.load(f)["zip_path"]

print(f"API key: {API_KEY}")
print(f"Model zip: {ZIP_PATH}")

In [ ]:
# ── Open session and import the labelling model ───────────────────────────────
config  = SessionConfig(name="risk-score-run")
session = client.open_session(
    api_key      = API_KEY,
    config       = config,
    artifact_dir = ARTIFACT_DIR,
)
session.import_model(ZIP_PATH)
print("Session ready.  Auth ID:", session.auth_id)

In [ ]:
# ── Kernel-restart recovery — paste values from above ────────────────────────
# AUTH_ID      = "paste-auth-id-here"
# ARTIFACT_DIR = "artifacts/fixed_log/risk_score/"
#
# session = client.get_session(AUTH_ID, artifact_dir=ARTIFACT_DIR)
# session.status()

## Step 9 — `configure_training`

Uploads your historical log data and configures feature engineering and calibration.
The file should cover the same operational period used during labelling — ideally
several months of log history so the model can learn what normal activity looks like.

### Core parameters

These apply to both unsupervised and supervised runs.

| Parameter | What it does | Default |
|-----------|-------------|--------|
| `asset_id_column` | Column that uniquely identifies each asset (e.g. train number, machine ID) | — |
| `timestamp_column` | Column containing the event timestamp; used to build rolling time windows | — |
| `date_format` | `'infer'` lets pandas auto-detect the format. Pass a strftime string (e.g. `'%d/%m/%Y'`) if auto-detection fails. | `'infer'` |
| `rolling_window` | Days to aggregate events over. Larger = smoother but slower to react. Must be 1–30. | `7` |
| `rolling_feature_type` | How events are aggregated in the rolling window — see table below | `'sum'` |
| `quantile_for_thresholds` | Percentile of the training distribution used as threshold (unsupervised) or fallback (supervised). Must be in [0.5, 1.0). | `0.99` |

**`rolling_feature_type` options:**

| Value | Meaning | Available in |
|-------|---------|-------------|
| `'sum'` | Total event count in the rolling window | Both modes |
| `'flag'` | 1 if any event occurred in the window, 0 otherwise | Both modes |
| `'ewm'` | Exponentially weighted mean — more weight on recent days | Fixed-log only |
| `'all'` | Runs sum + flag + ewm simultaneously | Fixed-log only |

### Supervised parameters

These are only used when `SUPERVISED = True`.  They are ignored if `SUPERVISED = False`.

| Parameter | What it does | Default |
|-----------|-------------|--------|
| `failures_path` | Path to the failures CSV file | — |
| `failure_date_column` | Column in the failures CSV that contains the failure date | `'failure_date'` |
| `failure_date_format` | `'infer'` (assumes dayfirst=True) or explicit strftime (e.g. `'%d/%m/%Y'`) | `'infer'` |
| `prediction_horizon_days` | Days *before* a failure to label as positive. A row at `failure_date - 3d` with `horizon=14` is labelled positive. | `14` |
| `exclusion_days_after` | Days *after* a failure to drop entirely — the recovery period has ambiguous behaviour. | `7` |
| `target_imbalance_ratio` | Cap on negatives:positives after downsampling. `10` keeps at most 10 negatives per positive. `None` keeps all. | `10.0` |
| `sampling_strategy` | `'block'` samples contiguous calendar-day blocks (preserves local temporal structure). `'random'` samples individual rows. | `'block'` |
| `block_size_days` | Block size in calendar days when `sampling_strategy='block'`. | `14` |
| `n_candidates` | Number of threshold values evaluated per subsystem during AUC grid search. More = better but slower. | `200` |
| `fallback_quantile` | Quantile used when a subsystem has fewer than `min_positives` positive rows. | `0.95` |
| `min_positives` | Minimum positive rows needed to attempt AUC-based threshold training for a subsystem. | `5` |
| `lr_c` | Logistic regression regularisation strength `C`. Smaller value = stronger regularisation (less overfitting). | `0.5` |

### Validation rules

- `asset_id_column` and `timestamp_column` must exist in the uploaded file
- `failures_path` must be provided when `supervised=True`
- `failure_date_column` must exist in the failures CSV
- `rolling_window` must be an integer between 1 and 30
- `quantile_for_thresholds` must be a float in [0.5, 1.0)
- `rolling_feature_type` of `'ewm'` or `'all'` will be rejected for free-text sessions
- `sampling_strategy` must be `'block'` or `'random'`

> **Re-runnable** until `train_risk_model` starts.

In [ ]:
# ── Core params (both modes) ──────────────────────────────────────────────────
TRAINING_FILE           = "test_files/training_data.csv" # path to your training data
ASSET_ID_COLUMN         = "asset_id"
TIMESTAMP_COLUMN        = "timestamp"
DATE_FORMAT             = "%d/%m/%Y"   # 'infer' to auto-detect
ROLLING_WINDOW          = 7            # days (1–30)
ROLLING_FEATURE_TYPE    = "sum"        # 'sum' | 'flag' | 'ewm' | 'all'  (fixed-log supports all four)
QUANTILE_FOR_THRESHOLDS = 0.99         # used as threshold (unsupervised) or fallback (supervised)
TRAINING_SHEET          = None         # XLSX only; None for CSV

# ── Mode selector ─────────────────────────────────────────────────────────────
SUPERVISED              = False        # ← set True to enable supervised calibration

# ── Supervised params (only used when SUPERVISED = True) ──────────────────────
FAILURES_FILE           = "test_files/failures.csv"   # CSV with asset_id + failure date
FAILURE_DATE_COLUMN     = "failure_date"
FAILURE_DATE_FORMAT     = "infer"      # 'infer' (dayfirst=True) or explicit e.g. '%d/%m/%Y'
PREDICTION_HORIZON_DAYS = 14           # days before failure → labelled positive
EXCLUSION_DAYS_AFTER    = 7            # days after failure → excluded (recovery window)
TARGET_IMBALANCE_RATIO  = 10.0         # max negatives:positives; None = keep all
SAMPLING_STRATEGY       = "block"      # 'block' (temporal) | 'random'
BLOCK_SIZE_DAYS         = 14
N_CANDIDATES            = 200          # threshold grid points per subsystem
FALLBACK_QUANTILE       = 0.95         # used when a subsystem has too few positives
MIN_POSITIVES           = 5            # min positive rows to attempt AUC training
LR_C                    = 0.5          # logistic regression regularisation (lower = stronger)

training_config_result = session.configure_training(
    file_path               = TRAINING_FILE,
    asset_id_column         = ASSET_ID_COLUMN,
    timestamp_column        = TIMESTAMP_COLUMN,
    date_format             = DATE_FORMAT,
    rolling_window          = ROLLING_WINDOW,
    rolling_feature_type    = ROLLING_FEATURE_TYPE,
    quantile_for_thresholds = QUANTILE_FOR_THRESHOLDS,
    sheet_name              = TRAINING_SHEET,
    # supervised params:
    supervised              = SUPERVISED,
    failures_path           = FAILURES_FILE if SUPERVISED else None,
    failure_date_column     = FAILURE_DATE_COLUMN,
    failure_date_format     = FAILURE_DATE_FORMAT,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    exclusion_days_after    = EXCLUSION_DAYS_AFTER,
    target_imbalance_ratio  = TARGET_IMBALANCE_RATIO,
    sampling_strategy       = SAMPLING_STRATEGY,
    block_size_days         = BLOCK_SIZE_DAYS,
    n_candidates            = N_CANDIDATES,
    fallback_quantile       = FALLBACK_QUANTILE,
    min_positives           = MIN_POSITIVES,
    lr_c                    = LR_C,
)
print(json.dumps(training_config_result, indent=2))

## Step 10 — `train_risk_model`

Runs the training pipeline on the server in the background.
The SDK polls until complete and prints rotating progress messages.
**Typical runtime: 1–15 minutes** depending on dataset size and mode.

### What happens server-side

**Both modes:**

1. **Feature engineering** — classifies logs against the hierarchy (Qdrant for free-text,
   exact-match for fixed-log) then builds rolling time-window features, producing one
   `_risk_feature` column per subsystem per (asset, date).

**Unsupervised path** (`SUPERVISED = False`):

2. **Threshold calibration** — for each subsystem the `quantile_for_thresholds` percentile
   of the `_risk_feature` distribution (across all assets and dates) is computed and
   stored as that subsystem's calibration threshold.
3. **Training data scoring** — the training dataset is scored with the calibrated thresholds
   and expert weights from the labelling step, producing `training_scores.parquet`.

**Supervised path** (`SUPERVISED = True`):

2. **Label attachment** — for each failure event, rows in `[failure_date - horizon, failure_date)`
   are labelled positive; rows in `[failure_date, failure_date + exclusion]` are dropped;
   all remaining rows are labelled negative.  The dataset is then downsampled.
3. **Threshold training** — for each subsystem a grid of `n_candidates` threshold values
   is evaluated; the one that maximises AUC-ROC on the training set is kept.
   If fewer than `min_positives` positive rows exist, `fallback_quantile` is used instead.
4. **Weight training** — a logistic regression is fitted on the calibrated risk features
   to learn system and subsystem weights.  Expert weights from the labelling step are
   used as a fallback for any system whose logistic regression coefficient is zero.
5. **Evaluation** — the full labeled dataset (no downsampling) is scored with both the
   baseline config (quantile thresholds + expert weights) and the supervised config.
   AUC-ROC and Average Precision are stored in `supervised_training_report.json`.
6. **Training data scoring** — the training dataset is scored with the trained thresholds
   and weights to produce `training_scores.parquet`.

### Files downloaded to `ARTIFACT_DIR` automatically

| File | Contents | Present when |
|------|----------|-------------|
| `calibration_thresholds.json` | Calibrated threshold per subsystem | Always |
| `training_fe.parquet` | Feature-engineered training dataset | Always |
| `training_scores.parquet` | Training data scored with trained model | Always |
| `logs_by_system_subsystem.json` | Log-to-subsystem mapping | Fixed-log only |
| `trained_weights.json` | System and subsystem weights from logistic regression | Supervised only |
| `supervised_training_report.json` | Evaluation metrics and dataset statistics | Supervised only |

### Kernel restart recovery

If your kernel restarts while training is running, check `session.status()` first.
If the step is `DONE` you can skip re-running and load from the saved artifact:

```python
with open(f'{ARTIFACT_DIR}train_risk_model.json') as f:
    training_result = json.load(f)
```

> **Re-runnable** until `configure_generation` starts.

In [ ]:
training_result = session.train_risk_model(timeout=3600)
print(json.dumps({k: v for k, v in training_result.items() if 'path' not in k}, indent=2))

In [ ]:
# ── Supervised training report (only populated when SUPERVISED = True) ────────
report_path = training_result.get("supervised_report_path")
if report_path:
    with open(report_path) as f:
        report = json.load(f)
    ev = report["evaluation"]
    print(f"Failures used:        {report['n_failures_used']}")
    print(f"Positive rows:        {report['n_positive_rows']} / {report['n_total_rows']} total")
    print(f"Subsystems trained:   {report['subsystems_trained']}")
    print()
    print(f"Baseline  AUC-ROC:        {ev.get('baseline_auc_roc', 'N/A')}")
    print(f"Supervised AUC-ROC:       {ev.get('supervised_auc_roc', 'N/A')}")
    print(f"Baseline  Avg Precision:  {ev.get('baseline_avg_precision', 'N/A')}")
    print(f"Supervised Avg Precision: {ev.get('supervised_avg_precision', 'N/A')}")
else:
    print("Unsupervised run — no training report.")

### Supervised training visualisations

*(Skip this section if `SUPERVISED = False`.)*

These helpers inspect the labeled dataset and compare baseline (quantile thresholds +
expert weights) against the supervised model (AUC thresholds + trained weights).
All accept file paths and work after a kernel restart.

| Helper | What it shows |
|--------|---------------|
| `plot_label_timeline` | Per-asset scatter coloured by label zone: blue = negative, red = positive (pre-failure), grey = excluded, stars = failure dates |
| `plot_label_distribution` | Stacked bar of positive vs negative row counts per week — shows whether positive signal is spread evenly over time |
| `plot_supervised_comparison` | Grouped bar chart: AUC-ROC and Average Precision for baseline vs supervised |
| `plot_weight_comparison` | Side-by-side bar + printed table comparing expert weights (from labelling) vs trained weights (from logistic regression) |

In [ ]:
# Label timeline — per-asset scatter coloured by label zone
helpers.plot_label_timeline(
    training_result["training_fe_path"],
    FAILURES_FILE,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    exclusion_days_after    = EXCLUSION_DAYS_AFTER,
    failure_date_column     = FAILURE_DATE_COLUMN,
)

# Label distribution — weekly positive vs negative row counts
helpers.plot_label_distribution(
    training_result["training_fe_path"],
    FAILURES_FILE,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    exclusion_days_after    = EXCLUSION_DAYS_AFTER,
    failure_date_column     = FAILURE_DATE_COLUMN,
    freq = "W",   # 'W' weekly | 'M' monthly
)

In [ ]:
# AUC-ROC and Average Precision: baseline vs supervised
helpers.plot_supervised_comparison(training_result["supervised_report_path"])

# Weight comparison: expert labelling weights vs trained logistic regression weights
#   system=None     → system-level comparison
#   system='name'   → subsystem-level comparison for that system
helpers.plot_weight_comparison(
    f"{ARTIFACT_DIR}model_config.json",
    training_result["trained_weights_path"],
    system = None,
)

### Training period evaluation

During `train_risk_model` the server scored the entire training dataset using the trained
thresholds and weights (for both supervised and unsupervised runs).  The result is saved
as `training_scores.parquet` and downloaded to `ARTIFACT_DIR` automatically.

Use these helpers to check how well the model captures risk before known failure events
*before* committing to scoring new data.  For unsupervised runs this acts as a sanity
check; for supervised runs it shows how the model performs on its own training period.

If you have no failures CSV, skip these two cells.

In [ ]:
training_scores_path = training_result.get("training_scores_path")
print(f"Training scores: {training_scores_path}")

# Risk time series around each failure event
# Dashed vertical line = failure date | Shaded region = prediction horizon
helpers.plot_risk_around_failures(
    training_scores_path,
    FAILURES_FILE,
    failure_date_column     = FAILURE_DATE_COLUMN,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    window_after_days       = 14,
    max_assets              = 6,
)

# Risk score distribution: pre-failure window vs normal operation + stats table
helpers.plot_failure_risk_stats(
    training_scores_path,
    FAILURES_FILE,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    exclusion_days_after    = EXCLUSION_DAYS_AFTER,
    failure_date_column     = FAILURE_DATE_COLUMN,
)

### Training feature visualisations

Use these helpers to sanity-check the calibrated thresholds before generating scores
on new data.  A threshold that sits near the mean of the distribution is too aggressive;
one that sits near the maximum may never be triggered.  All helpers accept file paths
and work after a kernel restart.

```python
# Fallback paths if training_result is not in memory (after kernel restart):
thresholds_path  = f"{ARTIFACT_DIR}calibration_thresholds.json"
training_fe_path = f"{ARTIFACT_DIR}training_fe.parquet"
```

| Helper | What it shows |
|--------|---------------|
| `plot_calibration_thresholds` | Bar chart: threshold vs mean and max of training distribution per subsystem |
| `plot_training_feature_distributions` | Histogram grid: one panel per subsystem with the calibrated threshold as a red line |
| `plot_training_feature_over_time` | Time series of a subsystem's rolling risk feature for selected assets with threshold as a dashed line |
| `plot_subsystem_feature_pipeline` | Three panels: risk feature / rolling features / binary presence — shows how the pipeline builds up for one asset/subsystem |

In [ ]:
thresholds_path   = training_result["thresholds_path"]
training_fe_path  = training_result["training_fe_path"]
logs_mapping_path = training_result.get("logs_mapping_path")   # fixed-log only

SYSTEMS_TO_INSPECT = None   # None = all systems; or e.g. ["motion_control", "power_supply"]

# 1. Threshold vs mean/max bar chart per subsystem
helpers.plot_calibration_thresholds(thresholds_path, training_fe_path, systems=SYSTEMS_TO_INSPECT)

# 2. Histogram of risk-feature distribution with calibrated threshold line
helpers.plot_training_feature_distributions(training_fe_path, thresholds_path, systems=SYSTEMS_TO_INSPECT)

In [ ]:
INSPECT_SYSTEM  = "motion_control"      # ← edit to your system name
INSPECT_SUBSYS  = "drive_unit"          # ← edit to your subsystem name
INSPECT_ASSETS  = ["A001", "A002"]     # ← edit to your asset IDs

# 3. Time series of a subsystem's rolling risk feature for selected assets
helpers.plot_training_feature_over_time(
    training_fe_path, thresholds_path,
    asset_id  = INSPECT_ASSETS,
    system    = INSPECT_SYSTEM,
    subsystem = INSPECT_SUBSYS,
)

# 4. Three-panel pipeline view for one asset: risk feature / rolling features / binary presence
helpers.plot_subsystem_feature_pipeline(
    training_fe_path, thresholds_path,
    asset_id     = INSPECT_ASSETS[0],
    system       = INSPECT_SYSTEM,
    subsystem    = INSPECT_SUBSYS,
    is_free_text = training_config_result["is_free_text"],
    logs_mapping = logs_mapping_path,   # only used when is_free_text=False
)

## Step 11 — `configure_generation`

Upload the log file you want to score.  This can be the same dataset used for training
or a completely new batch of operational logs.  The file must have the same column
structure as the training file because it goes through the identical classification and
feature engineering pipeline.

### `weights_source` — which weights to use for scoring

| Value | Source | When to use |
|-------|--------|-------------|
| `'labelling'` | `model_config.json` — expert weights defined during the labelling step | Always valid; use for unsupervised runs or as a baseline comparison |
| `'supervised'` | `trained_weights.json` — weights learned from failure events by logistic regression | Only after `train_risk_model` completed with `supervised=True` |

Passing `'supervised'` after an unsupervised training run will return an error from the API.
The `WEIGHTS_SOURCE` variable below auto-selects based on the `SUPERVISED` flag you set
in Step 9 — edit it manually if you want to compare modes.

> **Re-runnable** until `generate_risk_scores` starts.

In [ ]:
GENERATION_FILE  = "test_files/generation_data.csv"            # path to new .csv data you want to score on
DATE_FORMAT_GEN  = "%d/%m/%Y"                                   # 'infer' to auto-detect
GENERATION_SHEET = None                                         # XLSX only; None for CSV
WEIGHTS_SOURCE   = "supervised" if SUPERVISED else "labelling"  # auto-selects based on mode

generation_config_result = session.configure_generation(
    file_path      = GENERATION_FILE,
    date_format    = DATE_FORMAT_GEN,
    sheet_name     = GENERATION_SHEET,
    weights_source = WEIGHTS_SOURCE,
)
print(json.dumps(generation_config_result, indent=2))

## Step 12 — `generate_risk_scores` 🔒 ONE-WAY DOOR

**Cannot be repeated on the same session.**

Generates per-asset, per-period risk scores for every row in the generation dataset.

### What the SDK does automatically

1. Triggers generation on the server (runs in the background)
2. Polls until complete — prints rotating progress messages
3. **Downloads** `complete_model_{auth_id}.zip` into `OUTPUT_DIR`
4. **Extracts** key artifacts into `ARTIFACT_DIR`:
   - `risk_scores.parquet` → `result['parquet_path']`
   - `classified_logs.parquet` → `result['classified_logs_path']` *(free-text only)*
   - `logs_by_system_subsystem.json` → `result['logs_mapping_path']`
   - `calibration_thresholds.json` → `result['thresholds_path']`
   - `accepted_hierarchy.json` → `result['hierarchy_path']`
   - `model_config.json` → `result['model_config_path']`
5. **Wipes** the server session

### After a kernel restart

All plot helpers accept file paths — you do not need to re-run generation:

```python
parquet_path   = f"{ARTIFACT_DIR}risk_scores.parquet"
WEIGHTS_CONFIG = f"{ARTIFACT_DIR}model_config.json"   # or trained_weights.json
helpers.plot_asset_risk_ranking(parquet_path)
```

### To re-score new data using the same trained model

The `complete_model_ses-xxx.zip` downloaded to `OUTPUT_DIR` contains the full trained
model.  Open a new session and restore from it — this unlocks Step 11 directly,
skipping Steps 9–10:

```python
session.import_model("outputs/complete_model_ses-xxx.zip")
# Then go directly to configure_generation → generate_risk_scores
```

For a dedicated notebook that does this see `03_continuous_scoring.ipynb`.

In [ ]:
OUTPUT_DIR = "outputs/"   # ← directory to save the complete model zip into

generation_result = session.generate_risk_scores(OUTPUT_DIR, timeout=3600)

parquet_path = generation_result.get("parquet_path")
print(f"Risk scores:   {parquet_path}")
print(f"Assets scored: {generation_result['assets_scored']}")
print(f"Date range:    {generation_result['date_range']}")
print("Session wiped — use the zip in OUTPUT_DIR to restart.")

In [ ]:
# Resolve which weights were used — ensures weighted plots match how scores were generated
_trained_w = training_result.get("trained_weights_path")
WEIGHTS_CONFIG = (
    _trained_w
    if _trained_w and os.path.exists(_trained_w)
    else f"{ARTIFACT_DIR}model_config.json"
)
print(f"Weights config: {WEIGHTS_CONFIG}")

## Risk score visualisations

All helpers below accept either a file path or a DataFrame — the file path form works
after a kernel restart without re-running generation.

```python
# After kernel restart — set these manually:
parquet_path   = f"{ARTIFACT_DIR}risk_scores.parquet"
WEIGHTS_CONFIG = f"{ARTIFACT_DIR}model_config.json"   # or trained_weights.json if supervised
```

| Helper | What it shows | Mode |
|--------|--------------|------|
| `plot_asset_risk_ranking` | Bar chart: assets ranked by aggregated operational risk | Both |
| `plot_risk_score_distribution` | Histogram of risk scores across all records | Both |
| `plot_risk_heatmap` | Heatmap: assets × dates, coloured by chosen risk metric | Both |
| `plot_daily_risk_snapshot` | Tile panel for a single date — one tile per asset, system sub-tiles below | Both |
| `plot_asset_risk_over_time` | Heatmap of all risk types for one asset over time (raw + weighted) | Both |
| `plot_system_contributions` | Stacked weighted system contributions + operational risk line | Both |
| `plot_asset_system_over_time` | Subsystem heatmap for one system (raw + weighted) | Both |
| `plot_subsystem_log_activity` | Rolling features + binary presence + log counts per subsystem/asset | Both |
| `get_logs_for_subsystem` | Get specific log entries associated to the subsystem | Free text |

In [ ]:
# Fleet-wide ranking: top-N assets by aggregated operational risk
helpers.plot_asset_risk_ranking(
    parquet_path,
    aggregation = "max",   # 'mean' | 'max' | 'min' | 'median'
    top_n       = 20,
)

# Distribution of risk scores across all records
helpers.plot_risk_score_distribution(parquet_path)

In [ ]:
# Multi-asset risk heatmap over time
# metric= accepts 'operational_risk' or any '{system}_system_risk' column
print("Available risk metrics:")
for m in helpers.list_risk_metrics(parquet_path): print(f"  {m}")

ASSET_IDS = None   # None = all assets; or e.g. ["A001", "A002"]
helpers.plot_risk_heatmap(parquet_path, metric="operational_risk", asset_ids=ASSET_IDS)

# Single-date tile panel — one operational-risk tile per asset; system sub-tiles below
# Colour: blue (0–33), yellow (33–66), red (66–100)
SCORE_DATE = "2024-01-15"   # ← edit
helpers.plot_daily_risk_snapshot(parquet_path, date=SCORE_DATE)

In [ ]:
ASSET_TO_INSPECT = "A001"           # ← edit
DRILL_SYSTEM     = "motion_control" # ← edit to any system in your hierarchy

# All risk types for one asset over time
helpers.plot_asset_risk_over_time(parquet_path, asset_id=ASSET_TO_INSPECT)

# Weighted view — system rows scaled by their model weight
helpers.plot_asset_risk_over_time(
    parquet_path, asset_id=ASSET_TO_INSPECT, weighted=True, model_config=WEIGHTS_CONFIG,
    title=f"Risk Analysis — Weighted — {ASSET_TO_INSPECT}",
)

# Stacked weighted system contributions + operational risk line
helpers.plot_system_contributions(parquet_path, asset_id=ASSET_TO_INSPECT, model_config=WEIGHTS_CONFIG)

# Subsystem heatmap for one system (raw)
helpers.plot_asset_system_over_time(parquet_path, asset_id=ASSET_TO_INSPECT, system=DRILL_SYSTEM)

# Subsystem heatmap for one system (weighted)
helpers.plot_asset_system_over_time(
    parquet_path, asset_id=ASSET_TO_INSPECT, system=DRILL_SYSTEM,
    weighted=True, model_config=WEIGHTS_CONFIG,
)

In [ ]:
# Log-level drill-down: rolling features + binary presence + log counts
# Free-text mode → uses classified_logs.parquet
# Fixed-log mode → uses risk_scores.parquet + logs_by_system_subsystem.json
SUBSYSTEM  = "drive_unit"      # ← edit
END_DATE   = "2024-01-15"      # ← edit: plot the window ending on this date
DAYS_BACK  = 60
LOG_COLUMN = "event_details"   # ← your log text column name

classified_logs_path = generation_result.get("classified_logs_path")
logs_mapping_path    = generation_result.get("logs_mapping_path")

if classified_logs_path and os.path.exists(classified_logs_path):
    # Free-text mode
    helpers.plot_subsystem_log_activity(
        classified_logs_path,
        asset_id=ASSET_TO_INSPECT, system=DRILL_SYSTEM, subsystem=SUBSYSTEM,
        date=END_DATE, days_back=DAYS_BACK, log_column=LOG_COLUMN,
        risk_source=parquet_path,
    )
elif parquet_path and logs_mapping_path:
    # Fixed-log mode
    helpers.plot_subsystem_log_activity(
        parquet_path,
        asset_id=ASSET_TO_INSPECT, system=DRILL_SYSTEM, subsystem=SUBSYSTEM,
        date=END_DATE, days_back=DAYS_BACK, log_column=LOG_COLUMN,
        risk_source=parquet_path, logs_mapping=logs_mapping_path, is_free_text=False,
    )

In [ ]:
## for free-text data, get the associated underlying logs 

if not os.path.exists(classified_logs_path):
    print("classified_logs.parquet not available — this helper is for free-text mode only.")
    print(f"Expected at: {classified_logs_path}")
else:
    logs_df = helpers.get_logs_for_subsystem(
        classified_logs_path,
        asset_id=ASSET_TO_INSPECT,
        system=DRILL_SYSTEM,
        subsystem=SUBSYSTEM,
        date=END_DATE,
        days_back=DAYS_BACK,
        log_column=LOG_COLUMN,
    )
    print(f"{len(logs_df)} log entries found")
    display(logs_df)

## Failure performance on newly scored data

*(Optional — skip if you have no failures CSV or if the generation period does not overlap
with any failure events.)*

These helpers evaluate whether the model flags elevated risk in the period before known
failure events in the *generation* output — not the training period.
This is the key test of whether the model generalises beyond the data it was calibrated on.

If the generation dataset does not overlap with dates in your failures CSV the plots will
be empty.  In that case use the training-period evaluation cells above instead.

In [ ]:
# Risk time series around each failure event in the generation period
# Dashed vertical line = failure date | Shaded region = prediction horizon
helpers.plot_risk_around_failures(
    parquet_path,
    FAILURES_FILE,
    failure_date_column     = FAILURE_DATE_COLUMN,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    window_after_days       = 14,
    max_assets              = 6,
)

# Risk score distribution: pre-failure window vs normal operation + summary stats table
helpers.plot_failure_risk_stats(
    parquet_path,
    FAILURES_FILE,
    prediction_horizon_days = PREDICTION_HORIZON_DAYS,
    exclusion_days_after    = EXCLUSION_DAYS_AFTER,
    failure_date_column     = FAILURE_DATE_COLUMN,
)